In [1]:
library(tidyverse)
library(tidymodels)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
── Attaching packages ────────────────────────────────────── tidymodels 1.1.1 ──

✔ broom        1.0.6     ✔ rsample      1.2.1
✔ dials        1.3.0     ✔ tune         1.1.2
✔ infer        1.0.7     ✔ workflows    1.1.4
✔ modeldata    1.4.0     ✔ workflowsets 1.0.1
✔ parsnip      1.2.1     ✔ yardstick    1.3.1
✔ recipes      1.1.0     

── Conflicts ───────────────────────────────────────── tidymodels_conflicts() ──
✖ scales::discard() masks purrr::discard()
✖ dplyr::filt

In [43]:
players <- read_csv("../data/players.csv")

head(players)

Rows: 196 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): experience, hashedEmail, name, gender
dbl (2): played_hours, Age
lgl (1): subscribe

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


experience,subscribe,hashedEmail,played_hours,name,gender,Age
<chr>,<lgl>,<chr>,<dbl>,<chr>,<chr>,<dbl>
Pro,TRUE,f6daba428a5e19a3d47574858c13550499be23603422e6a0ee9728f8b53e192d,30.3,Morgan,Male,9
Veteran,TRUE,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa939732842f2312358a88e9,3.8,Christian,Male,17
Veteran,FALSE,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3c5a9d2118eb7ccbb28,0.0,Blake,Male,17
Amateur,TRUE,23fe711e0e3b77f1da7aa221ab1192afe21648d47d2b4fa7a5a659ff443a0eb5,0.7,Flora,Female,21
Regular,TRUE,7dc01f10bf20671ecfccdac23812b1b415acd42c2147cb0af4d48fcce2420f3e,0.1,Kylie,Male,21
Amateur,TRUE,f58aad5996a435f16b0284a3b267f973f9af99e7a89bee0430055a44fa92f977,0.0,Adrian,Female,17


In [44]:
players <- players |>
                mutate(experience = factor(experience, levels = c("Beginner", "Amateur", "Regular", "Veteran", "Pro"))) |>
                filter(!is.na(Age)) |>
                mutate(ages_0_10 = as.numeric(Age < 10)) |>
                mutate(ages_10_15 = as.numeric(10 <= Age & Age < 15)) |>
                mutate(ages_15_20 = as.numeric(15 <= Age & Age < 20)) |>
                mutate(ages_20_25 = as.numeric(20 <= Age & Age < 25)) |>
                mutate(ages_25_30 = as.numeric(25 <= Age & Age < 30)) |>
                mutate(ages_30plus = as.numeric(30 <= Age)) |>
                select(-hashedEmail, -name, -gender, -Age) |>
                mutate(subscribe = as.numeric(subscribe)) |>
                mutate(experience = as.numeric(experience))

players

experience,subscribe,played_hours,ages_0_10,ages_10_15,ages_15_20,ages_20_25,ages_25_30,ages_30plus
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
5,1,30.3,1,0,0,0,0,0
4,1,3.8,0,0,1,0,0,0
4,0,0.0,0,0,1,0,0,0
2,1,0.7,0,0,0,1,0,0
3,1,0.1,0,0,0,1,0,0
2,1,0.0,0,0,1,0,0,0
3,1,0.0,0,0,1,0,0,0
2,0,0.0,0,0,0,1,0,0
2,1,0.1,0,0,0,0,0,1


In [56]:
players_split <- initial_split(players, prop = 0.75)
players_train <- training(players_split)
players_test <- testing(players_split)

lr_spec = linear_reg() |>
            set_engine("lm") |>
            set_mode("regression")

recipe <- recipe(played_hours ~ experience + subscribe + ages_30plus + ages_15_20 + ages_20_25 + ages_25_30 + ages_0_10, data = players_train)

players_fit <- workflow() |>
                add_model(lr_spec) |>
                add_recipe(recipe) |>
                fit(data = players_train)

players_fit
                

══ Workflow [trained] ══════════════════════════════════════════════════════════
Preprocessor: Recipe
Model: linear_reg()

── Preprocessor ────────────────────────────────────────────────────────────────
0 Recipe Steps

── Model ───────────────────────────────────────────────────────────────────────

Call:
stats::lm(formula = ..y ~ ., data = data)

Coefficients:
(Intercept)   experience    subscribe  ages_30plus   ages_15_20   ages_20_25  
      2.411       -0.764        4.761       -0.400        2.746       -1.855  
 ages_25_30    ages_0_10  
     -2.426       26.948  


In [57]:
lm_test_results <- players_fit |>
  predict(players_test) |>
  bind_cols(players_test) |>
  metrics(truth = played_hours, estimate = .pred)

lm_test_results

.metric,.estimator,.estimate
<chr>,<chr>,<dbl>
rmse,standard,40.303781729
rsq,standard,0.003200806
mae,standard,13.661540040
